# Word Embeddings, Explained Simply
Turning words into numbers that carry meaning: one-hot vectors, cosine similarity, and training word2vec (skip-gram) from scratch in PyTorch. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
9,000 short, simple sentences (made up for this video) about animals, fruit, colours, vehicles and people.

### 9,000 simple sentences
- `urlretrieve`: download the file into Colab, unless it is already there
- `open(...).read()`: the whole file as one long string
- `line.split()`: split each sentence into words

In [ ]:
import os, urllib.request

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "word-embeddings/sentences.txt")
if not os.path.exists("sentences.txt"):
    urllib.request.urlretrieve(url, "sentences.txt")
text = open("sentences.txt").read()
sentences = [line.split() for line in text.strip().split("\n")]
print(len(sentences), "sentences")
for s in sentences[:4]:
    print(" ".join(s))

### The vocabulary
- `vocab`: every different word, sorted
- `index`: a lookup from each word to its number

In [ ]:
vocab = sorted({w for s in sentences for w in s})
index = {w: i for i, w in enumerate(vocab)}
print(len(vocab), "different words")
print("cat is word number", index["cat"])

## 2. One-hot vectors and cosine similarity
A **one-hot vector** is all zeros with a single 1. Every pair of different words is the same distance apart, so it carries no meaning. **Cosine similarity** = (a . b) / (|a| x |b|): 1 means pointing the same way, 0 means unrelated.

### One-hot: no meaning
- `one_hot`: 120 numbers: all zeros except a 1 at the word's position
- `torch.dist`: the straight-line distance between two vectors
- `the output`: 1.414 both times: cat is as far from dog as from car

In [ ]:
import torch

def one_hot(word):
    v = torch.zeros(len(vocab))
    v[index[word]] = 1
    return v

print("cat to dog:", torch.dist(one_hot("cat"), one_hot("dog")).item())
print("cat to car:", torch.dist(one_hot("cat"), one_hot("car")).item())

### Cosine similarity by hand
- `[0.9, 0.8]`: a made-up two-number embedding
- `F.cosine_similarity`: (a . b) / (|a| x |b|)
- `the output`: cat and dog point almost the same way; cat and car don't

In [ ]:
import torch.nn.functional as F

cat = torch.tensor([0.9, 0.8])
dog = torch.tensor([0.8, 0.9])
car = torch.tensor([0.9, -0.6])
print("cat-dog:", round(F.cosine_similarity(cat, dog, dim=0).item(), 3))
print("cat-car:", round(F.cosine_similarity(cat, car, dim=0).item(), 3))

## 3. Learning embeddings: skip-gram
Words used in the same places mean similar things. **Skip-gram** learns embeddings by predicting each word's neighbours (here, up to 2 words either side). Real (word, neighbour) pairs should score high; random pairs (**negative samples**) should score low. Score = dot product.

### Word-neighbour pairs
- `range(i - 2, i + 3)`: the two words before and the two after
- `if i != j`: not the word itself
- `the output`: about 145,000 pairs

In [ ]:
pairs = []
for s in sentences:
    for i in range(len(s)):
        for j in range(max(0, i - 2), min(len(s), i + 3)):
            if i != j:
                pairs.append((index[s[i]], index[s[j]]))
pairs = torch.tensor(pairs)
print(len(pairs), "pairs")
print([(vocab[a], vocab[b]) for a, b in pairs[:4].tolist()])

### Two embedding tables
- `nn.Embedding(120, 16)`: a table: 16 learnable numbers for each of the 120 words
- `ctx_emb`: a second table for words when they are neighbours
- `cat starts as`: random numbers, before any training
- `BCEWithLogitsLoss`: sigmoid plus log loss: push real scores up (to 1), random scores down (to 0)

In [ ]:
import torch.nn as nn

torch.manual_seed(0)
word_emb = nn.Embedding(len(vocab), 16)
ctx_emb = nn.Embedding(len(vocab), 16)
params = list(word_emb.parameters()) + list(ctx_emb.parameters())
optimizer = torch.optim.Adam(params, lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()
print(word_emb.weight.shape)
start = word_emb(torch.tensor(index["cat"]))[:4]
print("cat starts as:", [round(x, 2) for x in start.tolist()])

### Train
- `split(256)`: shuffled mini-batches of 256 pairs
- `neg`: 5 random words per pair: the negative samples
- `(word_emb(w) * ctx_emb(c)).sum(1)`: the dot product: the score for each real pair
- `loss_fn(pos, ...) + loss_fn(negs, ...)`: real pairs towards 1, random pairs towards 0

In [ ]:
for epoch in range(5):
    total = 0
    for b in torch.randperm(len(pairs)).split(256):
        w, c = pairs[b, 0], pairs[b, 1]
        neg = torch.randint(0, len(vocab), (len(b), 5))
        pos = (word_emb(w) * ctx_emb(c)).sum(1)
        negs = (word_emb(w).unsqueeze(1) * ctx_emb(neg)).sum(2)
        loss = (loss_fn(pos, torch.ones_like(pos))
                + loss_fn(negs, torch.zeros_like(negs)))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(b)
    print(f"epoch {epoch + 1}: loss {total / len(pairs):.3f}")

## 4. What did it learn?
Compare every word with cosine similarity, draw the words in 2D, and try word arithmetic.

### Most similar words
- `F.normalize`: make every embedding length 1, so a dot product is the cosine similarity
- `E @ E[index[word]]`: the word's similarity to all 120 words at once
- `[1:n + 1]`: skip the first: the word itself

In [ ]:
E = F.normalize(word_emb.weight.detach(), dim=1)

def similar(word, n=4):
    scores = E @ E[index[word]]
    best = scores.argsort(descending=True)[1:n + 1]
    return [(vocab[i], round(scores[i].item(), 2)) for i in best]

for word in ["cat", "mango", "red", "bus", "king"]:
    print(f"{word:<6}", similar(word))

### Draw the words in 2D
- `PCA(n_components=2)`: squash 16 numbers down to 2, keeping as much spread as possible
- `plt.annotate`: write each word next to its dot
- `the picture`: four separate clusters, learned from nothing but neighbours

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

groups = {"animal": ["cat", "dog", "rabbit", "horse", "cow", "sheep"],
          "fruit": ["apple", "mango", "banana", "orange", "grape", "cherry"],
          "colour": ["red", "blue", "green", "yellow", "purple", "pink"],
          "vehicle": ["car", "bus", "train", "bike", "truck", "boat"]}
words = sum(groups.values(), [])
xy = PCA(n_components=2).fit_transform(E[[index[w] for w in words]])
plt.figure(figsize=(12, 6))
for k, (g, ws) in enumerate(groups.items()):
    pts = xy[6 * k:6 * k + 6]
    plt.scatter(pts[:, 0], pts[:, 1], label=g)
    for (x, y), w in zip(pts, ws):
        plt.annotate(w, (x, y))
plt.legend(); plt.show()

### Word arithmetic
- `king - man + woman`: start at king, take away man, add woman
- `the loop`: the closest word that isn't one of the three we gave

In [ ]:
def analogy(a, b, c):
    v = E[index[b]] - E[index[a]] + E[index[c]]
    scores = E @ F.normalize(v, dim=0)
    for i in scores.argsort(descending=True):
        if vocab[i] not in (a, b, c):
            return vocab[i]

print("man : king  = woman :", analogy("man", "king", "woman"))
print("boy : prince = girl :", analogy("boy", "prince", "girl"))
print("man : boy   = woman :", analogy("man", "boy", "woman"))

## 5. Mini project: odd one out
Find the word that is least similar to the others.

### Odd one out
- `vecs @ vecs.T`: every word's similarity to every other
- `mean(1)`: each word's average similarity
- `argmin()`: the word that fits in least

In [ ]:
def odd_one_out(words):
    vecs = E[[index[w] for w in words]]
    avg_sim = (vecs @ vecs.T).mean(1)
    return words[avg_sim.argmin()]

print(odd_one_out(["cat", "dog", "mango", "cow"]))
print(odd_one_out(["red", "bus", "train", "bike"]))
print(odd_one_out(["queen", "princess", "girl", "truck"]))

## Try it yourself
1. Work out the cosine similarity of dog and car by hand. Then check it with the code.
2. Change the embedding size from 16 to 2. Do the similar words still make sense?
3. Try the analogy boy is to man as girl is to ... ?